# Scaffold Analysis — Frequency & Mean Z-Score

Computes Murcko scaffolds for each molecule, aggregates frequency and mean consensus z-score per scaffold, and exports to CSV.

In [44]:
CSV_PATH     = '../Ensemble/all_consensus_zscore.csv'
SMILES_COL   = 'SMILES'
ZSCORE_COL   = 'consensus_z'
ID_COL       = 'ID'

OUTPUT_CSV   = '../Results/scaffold_analysis.csv'

# Bayesian shrinkage parameter k.
# None = estimated from data as mean scaffold size (recommended).
# Set a fixed int to override (e.g. k=5 means a scaffold needs ~5 compounds
# to contribute half its raw mean; smaller k = less shrinkage).
SHRINKAGE_K  = None

In [45]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

In [46]:
df = pd.read_csv(CSV_PATH)
print(f'Loaded: {len(df)} rows | columns: {list(df.columns)}')

df[ZSCORE_COL] = pd.to_numeric(df[ZSCORE_COL], errors='coerce')
n_before = len(df)
df = df.dropna(subset=[SMILES_COL, ZSCORE_COL]).reset_index(drop=True)
print(f'{n_before - len(df)} rows removed (null SMILES/z-score). Remaining: {len(df)}')

df.head()

Loaded: 4398 rows | columns: ['ID', 'library', 'SMILES', 'Name', 'Formula', 'MW', 'LogP', 'pKd', 'score', 'z_pkd', 'z_vina', 'consensus_z', 'SA_Score', 'Structure', 'Synthesize', 'Chemical', 'Dataset']
0 rows removed (null SMILES/z-score). Remaining: 4398


,ID,library,SMILES,Name,Formula,MW,LogP,pKd,score,z_pkd,z_vina,consensus_z,SA_Score,Structure,Synthesize,Chemical,Dataset
0,Druglike_1,Druglike,NC1=C2C=NNC2=NC([C@@H](/C=C/CC2=NC(CCCl)=CC=C2...,21/000000020,C17H17N6O2Cl,372.0,2.17,6.45,-9.496,6.042542,1.343421,3.692981,3.895024,6.97,24.1,-190.0,DRUGLIKE_CONCAT
1,HighDiv_4,HighDiv,NOC1=CC(C(=O)O)=CC(OC/C=C/[C@@H]2NNC3=C2C=CC=C...,12/000001403,C17H16N3O4Br,406.0,2.98,5.86,-10.609,4.310268,2.570990,3.440629,3.709026,7.55,21.3,-190.0,HIGHDIVERSITY_CONCAT
2,HighDiv_1581,HighDiv,C=CC1=C(C(N)=O)SC2=CC=C([S@+]([O-])C3=CC(O)=C(...,12/000000150,C17H12NO4S2I,485.0,2.13,5.68,-10.932,3.781778,2.927239,3.354509,3.880718,6.18,24.8,-40.0,HIGHDIVERSITY_LANAPDB
3,HighDiv_1,HighDiv,C[C@H](CN)CC#CC1=CC(C(=O)CO)=CC(N=O)=C1Br,12/000000691,C14H16N2O3Br,340.0,1.76,6.10,-9.375,5.014922,1.209965,3.112444,4.386506,5.24,20.7,-170.0,HIGHDIVERSITY_CONCAT
4,Druglike_2,Druglike,NC1=C2C=NNC2=NC([C@@H](/C=C/CC2=NC=CC=C2CCBr)C...,21/000000030,C17H17N6O2Br,417.0,2.42,6.04,-9.526,4.838759,1.376509,3.107634,3.978020,5.01,24.1,-190.0,DRUGLIKE_CONCAT


In [47]:
def get_scaffold(smi):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    try:
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        return scaffold if scaffold else Chem.MolToSmiles(mol)
    except Exception:
        return Chem.MolToSmiles(mol)

df['scaffold'] = df[SMILES_COL].apply(get_scaffold)

n_invalid = df['scaffold'].isna().sum()
df = df.dropna(subset=['scaffold']).reset_index(drop=True)
print(f'{n_invalid} molecules with invalid SMILES removed. Remaining: {len(df)}')
print(f'{df["scaffold"].nunique()} unique Murcko scaffolds')

0 molecules with invalid SMILES removed. Remaining: 4398
1231 unique Murcko scaffolds


In [48]:
global_mean = df[ZSCORE_COL].mean()

scaffold_df = (
    df.groupby('scaffold')
    .agg(
        frequency   = (ZSCORE_COL, 'count'),
        mean_zscore = (ZSCORE_COL, 'mean'),
    )
    .reset_index()
    .sort_values('frequency', ascending=False)
    .reset_index(drop=True)
)

# Bayesian shrinkage: (n * mean_scaffold + k * global_mean) / (n + k)
# Scaffolds with n << k are pulled toward the global mean;
# scaffolds with n >> k converge to their raw mean.
k = SHRINKAGE_K if SHRINKAGE_K is not None else scaffold_df['frequency'].mean()
scaffold_df['shrunk_zscore'] = (
    (scaffold_df['frequency'] * scaffold_df['mean_zscore'] + k * global_mean)
    / (scaffold_df['frequency'] + k)
)

scaffold_df['mean_zscore']   = scaffold_df['mean_zscore'].round(4)
scaffold_df['shrunk_zscore'] = scaffold_df['shrunk_zscore'].round(4)

print(f'Global mean z-score : {global_mean:.4f}')
print(f'Shrinkage k         : {k:.2f}')
print(f'Scaffolds saved     : {len(scaffold_df)}')

scaffold_df.to_csv(OUTPUT_CSV, index=False, sep=';')
print(f'Saved to {OUTPUT_CSV}')
scaffold_df.head(10)

Global mean z-score : 0.0000
Shrinkage k         : 3.57
Scaffolds saved     : 1231
Saved to ../Results/scaffold_analysis.csv


,scaffold,frequency,mean_zscore,shrunk_zscore
0,c1ccccc1,941,-0.3581,-0.3567
1,O=C1NCCN1c1ccccc1,146,0.0341,0.0333
2,c1ccncc1,136,-0.5552,-0.5410
3,O=C(c1ccc(-c2ccccc2)[nH]1)c1nc(=O)o[nH]1,101,1.4772,1.4267
4,c1ccc(-c2cn[nH]n2)cc1,91,-0.1959,-0.1885
5,c1ccc(-c2c[nH]cn2)cc1,84,0.5882,0.5642
6,c1cncnc1,60,-0.6194,-0.5846
7,O=c1ccc(-c2cnco2)n[nH]1,57,-0.0016,-0.0015
8,O=c1cc[nH]c(-c2ccccc2)c1,51,0.2668,0.2493
9,c1cnoc1,51,-1.0001,-0.9347
